# Safe 3-way ensemble — seed42 screen

## 가설
동일한 규정 안전 structured/enrichment feature에서 multinomial LR, OVR LR, base LightGBM은 서로 다른 경계를 만들어 H0 오류를 보완할 수 있습니다. 가중치는 사전 고정된 0.55 / 0.30 / 0.15입니다.

## 규칙
- Stratified 5-fold, seed42, train만 읽음
- vocabulary·recurrent event·enrichment·자동 specialist는 outer-fold train에서만 fit
- test concat, test scaling/encoding/selection, 고정 gene/class/exact mutation 목록, blend 탐색 금지
- WT/blank/NaN은 event가 아니며 결과에서 NaN mutation 0을 검증

기존 팀 코드의 fixed hotspot/contrast 규칙은 사용하지 않습니다. 이 screen이 H0보다 +0.005 및 4/5 fold 상승일 때만 후속 3-seed 또는 fixed gate 결합을 검토합니다.

In [ ]:
from pathlib import Path
import subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
RUNNER = ROOT / 'experiments/gs/notebooks/exp_model_006/common/run_safe_3way_ensemble.py'
RESULT = ROOT / 'experiments/gs/notebooks/exp_model_006/result'
RUN_ID = 'exp-safe-3way-ensemble-01'
RUN_EXPERIMENT = True
assert RUNNER.exists()
print({'runner': RUNNER, 'result_dir': RESULT, 'seed': 42, 'test_read': False, 'weights': {'multinomial': .55, 'ovr': .30, 'lightgbm': .15}})

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='safe 3-way folds', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('Safe 3-way runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 기존 result만 읽습니다.')

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(RESULT / f'{RUN_ID}_seed42_summary.csv')
folds = pd.read_csv(RESULT / f'{RUN_ID}_seed42_fold_metrics.csv')
classes = pd.read_csv(RESULT / f'{RUN_ID}_seed42_class_metrics.csv')
audit = json.loads((RESULT / f'{RUN_ID}_seed42_leakage_audit.json').read_text())
assert summary.leakage_check.all() and summary.nan_as_mutation_count.eq(0).all()
display(summary.sort_values('oof_macro_f1', ascending=False))
print('자동 판정:', audit['decision'])
print({key: audit[key] for key in ('delta_vs_h0', 'positive_fold_count', 'h0_reference_delta', 'weights')})

folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', figsize=(10, 4), title='Safe 3-way: fold Macro F1')
plt.ylabel('Macro F1'); plt.tight_layout(); plt.show()
classes.set_index('class').delta.sort_values().plot.barh(figsize=(7, 7), title='Safe 3-way class F1 delta vs H0')
plt.tight_layout(); plt.show()